# Residual Stream Geometry

Analyze the geometric structure of the residual stream: angles, subspace dimension,
update geometry, pairwise distances, and geometry summary.

## Why This Matters

Residual stream stream geometry characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_geometry import (
    residual_angle_structure, residual_subspace_dimension,
    residual_update_geometry, residual_pairwise_distances,
    residual_geometry_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Angle Structure

How does the residual stream rotate relative to embedding and unembedding?

In [ ]:
result = residual_angle_structure(model, tokens, position=-1)
print(f"Target token: {result['target_token']}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: cos_embed={p['cosine_to_embed']:.4f}, "
          f"cos_unembed={p['cosine_to_unembed']:.4f}, "
          f"cos_prev={p['cosine_to_previous']:.4f}, norm={p['norm']:.4f}")

## Subspace Dimension

How many dimensions do residual vectors actually use?

In [ ]:
for layer in range(4):
    result = residual_subspace_dimension(model, tokens, layer=layer)
    print(f"  Layer {layer}: PR={result['participation_ratio']:.2f}, "
          f"dim90={result['dim_for_90_pct']}, dim95={result['dim_for_95_pct']}, "
          f"top_sv={result['top_sv_fraction']:.2%}")

## Update Geometry

Are layer updates parallel or perpendicular to the residual stream?

In [ ]:
result = residual_update_geometry(model, tokens, position=-1)
print(f"Mostly perpendicular layers: {result['mostly_perpendicular']}\n")
for p in result['per_layer']:
    flag = ' [PERP]' if p['is_mostly_perpendicular'] else ' [PAR]'
    print(f"  Layer {p['layer']}: update={p['update_norm']:.4f}, "
          f"parallel={p['parallel_magnitude']:+.4f}, "
          f"perp={p['perpendicular_magnitude']:.4f}{flag}")

## Pairwise Distances

How spread out are position representations?

In [ ]:
for layer in range(4):
    result = residual_pairwise_distances(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean={result['mean_distance']:.4f}, "
          f"min={result['min_distance']:.4f}, max={result['max_distance']:.4f}, "
          f"spread={result['spread_ratio']:.2f}")

## Geometry Summary

Cross-layer overview of residual stream geometry.

In [ ]:
result = residual_geometry_summary(model, tokens)
print(f"Norm trend: {result['norm_trend']}\n")
for p in result['per_layer']:
    bar = '█' * int(min(p['mean_norm'] * 5, 30))
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"PR={p['participation_ratio']:.2f} {bar}")